In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

c:\Users\Ansh\OneDrive\Desktop\Langraph\meraenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=os.getenv('GROQ_API_KEY'),
    temperature = 0.5
)

In [4]:
# creating a state

class LinkedIn(TypedDict):
    topic:str
    post:str
    feedback:str
    evaluation:str
    iteration:int
    max_iter:int


In [5]:
class Evaluation(BaseModel):
    evaluation:Literal['approved','rejected'] = Field(description="Whether the LinkedIn post is approved or rejected based on the feedback provided.")
    feedback:str = Field(description = "Feedback on the LinkedIn post.")

strucuted_model = llm.with_structured_output(Evaluation)  # this is our new model that gives only 2 things as output - evaluation and feedback nothing else

In [7]:
def generate(state:LinkedIn):
    prompt = f"""Create a LinkedIn post on the topic: {state['topic']}.
      Act as a professional tech content writer and LinkedIn thought leader.

    The topic for today’s post is: [USER-DEFINED TECH TOPIC]

    Write a high-quality, engaging LinkedIn post on this topic.

    Guidelines:

    Begin with a compelling hook in the first 1–2 lines.

    Explain the topic clearly so even non-experts can understand it.

    Emphasize why this topic is important or trending right now.

    Share one meaningful insight, lesson, or real-world application.

    Maintain a professional, confident, and conversational tone suitable for LinkedIn.

    Use short paragraphs and line breaks for easy readability.

    Use 2–3 relevant emojis sparingly.

    End with a question or call-to-action to encourage comments.

    Length: 120–180 words.

    Add 4–6 relevant hashtags at the end (no hashtags in the body)."""
        
    post = llm.invoke(prompt).content
    return {'post':post}

In [8]:
def evaluate(state:LinkedIn):
    prompt = f"""You are a senior content strategist reviewing a LinkedIn post draft.

    Evaluate the following LinkedIn post:
    \"\"\"
    {state['post']}
    \"\"\"

    Evaluate the post strictly based on the criteria below and decide whether the post should be
    **APPROVED** or **REJECTED**.

    Evaluation Parameters:
    1. Hook Strength – Does the opening grab attention immediately?
    2. Clarity & Readability – Is the message easy to understand and well-structured?
    3. Relevance to Topic – How well does the post stay aligned with the given topic?
    4. Value Provided – Does it offer insights, learning, or practical takeaways?
    5. Engagement Potential – Likelihood of likes, comments, and shares.
    6. Tone & Professionalism – Is the tone appropriate for LinkedIn?
    7. Emoji Usage – Are emojis relevant and used sparingly?
    8. Call-to-Action (CTA) – Is the CTA clear and effective?
    9. Hashtag Quality – Are hashtags relevant and well-chosen?

    Rejection Rules (VERY IMPORTANT):
    - Reject the post if **any** of the following conditions are met:
    - Hook Strength score < 7
    - Clarity & Readability score < 8
    - Value Provided score < 7
    - Engagement Potential score < 8
    - Tone & Professionalism score < 7
    - More than 3 emojis are used
    - No clear CTA is present
    - Hashtags are missing or irrelevant

    Scoring Instructions:
    - Score each parameter on a scale of 1–10.

    Final Output Requirements:
    - Provide:
    1. An  **evaluation** :  APPROVED or REJECTED
    2. If REJECTED, provide **concise, actionable feedback** explaining what must be improved
    - Do NOT rewrite the post.
    - Be strict and objective.
    """

    feedback = strucuted_model.invoke(prompt).feedback
    evaluation = strucuted_model.invoke(prompt).evaluation
    
    return {'feedback':feedback,'evaluation':evaluation}

In [9]:
def optimize(state:LinkedIn):
    prompt = f"""Act as a professional LinkedIn content strategist and editor.

    You are given an original LinkedIn post regarding topic: '{state['topic']}' and structured feedback from an evaluation.
    Your task is to rewrite and optimize the post by directly addressing the feedback, while preserving the original intent and topic.
    Improve the following LinkedIn post based on this feedback: {state['feedback']}.
    Post: {state['post']}
    Optimization Guidelines:
    Strengthen weak areas highlighted in the feedback (low scores first).
    Improve the hook if hook strength is weak.
    Enhance clarity and readability using short paragraphs.
    Add or refine a practical insight or takeaway if value is low.
    Increase engagement by improving the call-to-action.
    Maintain a professional, conversational LinkedIn tone.
    Use emojis only if they improve engagement (max 2–3).
    Keep the post between 120–180 words.
    Include 4–6 relevant hashtags at the end only.

    Output Rules:

    Return only the optimized LinkedIn post.
    Do not explain changes.
    Do not include feedback or scores in the output."""

    post = llm.invoke(prompt).content
    iteration = state['iteration'] + 1
    return {'post':post,'iteration':iteration}

In [12]:
def check_condition(state:LinkedIn)->Literal['end','optimize']:
    if state['evaluation'].lower() == 'approved' or state['iteration'] >= 5:
        return 'end'
    else:
        return 'optimize'

In [10]:
def end(state:LinkedIn):
    return {'post':state['post'],'iteration':state['iteration']}

In [15]:
graph = StateGraph(LinkedIn)   # LINKEDIN IS THE NAME OF STATE (DICT)

graph.add_node('generate',generate)
graph.add_node('evaluate',evaluate)
graph.add_node('optimize',optimize)
graph.add_node('end',end)

graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')
graph.add_conditional_edges('evaluate',check_condition)
graph.add_edge('optimize','evaluate')
graph.add_edge('end',END)




In [16]:
workflow = graph.compile()
workflow

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [108]:
# from IPython.display import Image
# Image(workflow.get_graph().draw_mermaid_png())

In [20]:
final_state = workflow.invoke({'topic':'HONDA CARS AS AN AI ENGINNEER', 'iteration': 0})
print(final_state['post'])
print(final_state['iteration'])

**Revolutionizing the Automotive Industry: How Honda's Innovations are Inspiring AI Engineers 🚗💻**

As an AI engineer, I'm always on the lookout for real-world applications that showcase the potential of artificial intelligence. One company that's making waves in this space is Honda, the renowned Japanese automaker. Their commitment to innovation has led to some groundbreaking advancements that are redefining the automotive industry.

Honda's focus on AI-powered technologies, such as predictive maintenance and autonomous driving, is not only enhancing the driving experience but also paving the way for a more sustainable future. By leveraging data analytics and machine learning algorithms, Honda is able to optimize vehicle performance, reduce energy consumption, and improve road safety.

What's more, Honda's innovations are inspiring a new generation of AI engineers to think creatively about the possibilities of AI in the automotive sector. As we continue to navigate the complexities of